## 1. 커스텀 Tool 설계 원칙

ReAct 에이전트에서 Tool은 에이전트가 외부 세계와 상호작용하는 유일한 통로이다. 따라서 Tool의 설계 품질이 에이전트의 전체 성능을 좌우한다. 다음은 커스텀 Tool을 설계할 때 반드시 지켜야 할 핵심 원칙들이다.

### 1.1 단일 책임 원칙 (Single Responsibility)

하나의 Tool은 **하나의 명확한 기능**만 수행해야 한다. 뉴스 검색과 요약을 하나의 Tool에 넣는 것이 아니라, 검색 Tool과 요약 Tool을 분리하여 구현한다. 이렇게 하면 LLM이 각 도구의 역할을 명확히 이해할 수 있고, 도구 조합의 유연성이 높아진다.

### 1.2 명확한 Tool 설명문 작성

Tool의 설명문(description)은 LLM이 도구를 올바르게 선택하는 데 결정적인 역할을 한다. 설명문에는 다음 요소가 포함되어야 한다:

- **기능 설명**: 이 도구가 무엇을 하는지 한 문장으로 명시
- **입력 형식**: 어떤 형식의 입력을 기대하는지 구체적으로 명시
- **사용 예시**: 실제 호출 예시를 포함하여 LLM이 패턴을 학습하도록 유도

### 1.3 입력/출력 형식 표준화

모든 Tool은 **문자열 입력 -> 문자열 출력** 형태를 유지하는 것이 좋다. ReAct 프레임워크에서 Action과 Observation은 텍스트 기반이므로, 일관된 문자열 인터페이스를 유지하면 에이전트와의 통합이 용이하다.

### 1.4 에러 처리와 Fallback 전략

외부 API를 호출하는 Tool은 반드시 예외 처리를 포함해야 한다. 네트워크 오류, 타임아웃, API 제한 등 다양한 실패 상황에 대비하여:

- try/except로 예외를 포착하고 의미 있는 오류 메시지를 반환
- 타임아웃 설정으로 무한 대기 방지
- 재시도 로직으로 일시적 오류에 대응
- 대체 데이터 소스나 기본값을 준비

## 2. 뉴스 검색 커스텀 Tool 개발

첫 번째 커스텀 Tool로 Google News RSS 피드를 활용한 **뉴스 검색 Tool**을 구현한다. 이 Tool은 실시간 뉴스 데이터를 수집하여 에이전트에게 최신 정보를 제공하는 역할을 한다.

### RSS 기반 뉴스 검색의 장점

- 별도의 API 키 없이 무료로 사용 가능
- XML 형식으로 구조화된 데이터 제공
- 실시간 최신 뉴스 접근 가능
- 한국어 뉴스 지원


- https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko

In [1]:
import os
import json
import requests
import xml.etree.ElementTree as ET
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def fetch_news(query, max_results=3):
    '''Goole News RSS에서 뉴스를 검색합니다.'''
    try:
        url = f'https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko'
        response = requests.get(url,timeout=10)        

        root = ET.fromstring(response.content)
        items = root.findall('.//item')[:max_results]  # .현재노드  // 모든 하위 태그탐색  item  <item>태그
        results = []
        for item in items:
            title = item.find('title').text if item.find('title') is not None else 'N/A'
            pub_date = item.find('pubDate').text if item.find('title') is not None else 'N/A'
            source = item.find('source').text if item.find('title') is not None else 'N/A'
            results.append({'title':title, 'date':pub_date, 'source':source})
        if results:
            output = f'{query}관련 최신 뉴스 {len(results)}건\n'
            for i, r in enumerate(results,1):
                output += f"  {i}. [{r['source']} {r['title']}  {r['date']}]"
            return output
        return f'{query}에 대한 뉴스를 찾을 수 없습니다'
    except Exception as e:
        return f'뉴스검색오류 : {e}'

print(fetch_news("인공지능"))

인공지능관련 최신 뉴스 3건
  1. [머니투데이 "AI가 스스로 해킹한다"…국내 대학도 털렸다 - 머니투데이 - 머니투데이  Thu, 28 May 2026 03:00:00 GMT]  2. [지디넷코리아 정부, 젊은 인재 키우는 'AI스타펠로우십' 재공고 - 지디넷코리아  Thu, 28 May 2026 02:55:16 GMT]  3. [v.daum.net "2년내 누구나 AI에이전트 만드는 시대 온다"…MS 부사장의 예언 - v.daum.net  Fri, 29 May 2026 01:26:05 GMT]


### 요약이나 분석은 별도의 tool에 위임

In [2]:
# 텍스트 요약
def summarize_text(text:str, max_length=100):
    try:
        response  = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = [
                {'role':'system','content':f'텍스트를 {max_length}자 이내로 간결하게 요약하는 시스템 입니다.'},
                {'role':'user','content':text}                
            ],
            temperature=0,
            max_completion_tokens=max_length+100
        )
        return f'요약 : {response.choices[0].message.content}'
    except Exception as e:
        return f'요약 오류 : {e}'

In [3]:
# tool chain 테스트
summarize_text(fetch_news("인공지능"))

'요약 : AI 해킹·대학 피해, 정부 AI 스타펠로우십 재공고, MS 부사장 “2년 내 누구나 AI 에이전트” 전망.'

## 3. 에러 핸들링이 포함된 견고한 ReAct 에이전트

이전 에서 구현한 기본 ReAct 에이전트는 에러 발생 시 바로 중단되는 문제가 있었다. 실무 환경에서는 네트워크 장애, API 제한, 잘못된 입력 등 다양한 예외 상황이 발생할 수 있으므로, 이를 우아하게 처리하는 **견고한(Robust) 에이전트**가 필요하다.

### RobustReActAgent의 핵심 기능

- **자동 재시도(Retry)**: Tool 실행 실패 시 설정된 횟수만큼 재시도
- **실행 로그(Execution Log)**: 모든 단계를 기록하여 디버깅과 분석 지원
- **형식 오류 복구**: LLM이 잘못된 형식으로 응답하면 재시도 요청
- **최대 단계 제한**: 무한 루프를 방지하는 안전장치

In [ ]:
import re
import time
from typing import Callable, Dict, Optional
class RobustReActAgent:
    def __init__(self,model = 'gpt-5.3-nana', max_steps=6, retry_count=2):
        self.model=model
        self.max_steps = max_steps
        self.retry_count = retry_count
        self.tools:Dict[str,Dict] = {}
        self.excution_log = []